# 01 — 카테고리 후보 EDA (Phase 01 step4)

wine21 50병 sample (`data/raw/wine21/20260514/records.jsonl`) 분포를 보고
MAB arm 으로 쓸 카테고리 축을 (지역×품종) vs (지역×가격대) 중에서 결정한다.

CLAUDE.md 규칙: notebook 은 호출·시각화 전용. matplotlib only.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

RECORDS = Path("../data/raw/wine21/20260514/records.jsonl")
records = [json.loads(line) for line in RECORDS.open(encoding="utf-8")]
print(f"{len(records)} records loaded")

In [ ]:
RED = {
    "카베르네 소비뇽",
    "피노 누아",
    "시라/쉬라즈",
    "그르나슈",
    "메를로",
    "카르메네르",
    "사페라비",
    "카베르네 프랑",
}
WHITE = {"샤르도네", "샤도네이", "소비뇽 블랑", "리슬링", "세미용", "피노 블랑", "그르나슈 블랑"}
SPARK = {"피노 뮈니에"}
COUNTRY_MAP = {
    "프랑스(France)": "프랑스",
    "미국(U.S.A)": "미국",
    "호주(Australia)": "호주",
    "칠레(Chile)": "칠레",
    "뉴질랜드(New Zealand)": "뉴질랜드",
}


def country(r):
    c = r["region"].split(">")[0].strip() if r["region"] else "기타"
    return COUNTRY_MAP.get(c, "기타")


def grape(r):
    return (r["varietal"] or "").split("(")[0].strip() or "기타"


def color(r):
    g = grape(r)
    if g in RED:
        return "레드"
    if g in WHITE:
        return "화이트"
    if g in SPARK:
        return "스파클링"
    return "기타"


def price_band(r):
    p = r["price_krw"]
    if p is None:
        return "가격미상"
    if p < 60000:
        return "저가"
    if p < 150000:
        return "중가"
    return "고가"

## 1. 지역별 빈도

In [ ]:
cc = Counter(country(r) for r in records)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(cc.keys()), list(cc.values()))
ax.set_title("지역별 빈도 (n=50)")
ax.set_ylabel("병 수")
plt.tight_layout()
plt.show()

## 2. 품종별 빈도

In [ ]:
gc = Counter(grape(r) for r in records).most_common()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh([g for g, _ in gc][::-1], [c for _, c in gc][::-1])
ax.set_title("품종별 빈도 (n=50)")
ax.set_xlabel("병 수")
plt.tight_layout()
plt.show()

## 3. 가격 분포 히스토그램

In [ ]:
prices = [r["price_krw"] for r in records if r["price_krw"] is not None]
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(prices, bins=[0, 60000, 150000, 300000, 1000000])
ax.set_title(f"가격 분포 (n={len(prices)})")
ax.set_xlabel("가격 (KRW)")
ax.set_ylabel("병 수")
plt.tight_layout()
plt.show()

## 4. (지역 × 품종색) 교차표 히트맵

In [ ]:
def heatmap(rows, cols, counter, title):
    grid = [[counter.get((row, col), 0) for col in cols] for row in rows]
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(grid, cmap="Blues")
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols)
    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows)
    for i in range(len(rows)):
        for j in range(len(cols)):
            ax.text(j, i, grid[i][j], ha="center", va="center")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


countries = ["프랑스", "미국", "호주", "칠레", "뉴질랜드", "기타"]
colors = ["레드", "화이트", "스파클링", "기타"]
cxv = Counter((country(r), color(r)) for r in records)
heatmap(countries, colors, cxv, "지역 × 품종색 (n=50)")

## 5. (지역 × 가격대) 교차표 히트맵

In [ ]:
bands = ["저가", "중가", "고가", "가격미상"]
cxb = Counter((country(r), price_band(r)) for r in records)
heatmap(countries, bands, cxb, "지역 × 가격대 (n=50)")

## 결론

- **지역 × 품종색**: 상위 셀(프랑스-레드 12)에 쏠리고, 단발(1병) 셀이 6개로 꼬리가 길다.
- **지역 × 가격대**: ≥2병 셀이 10개로 더 고르게 퍼진다. 가격은 "합리적 취향 탐색"의 핵심 결정 축이기도 함.
- → **카테고리 축은 (지역 × 가격대)** 로 확정. 10개 카테고리를 `data/processed/category_candidates.json` 에 저장.
- 가격대 임계: 저가 < 6만 / 중가 6만~15만 / 고가 ≥ 15만 (50병 사분위 58k·100k·198k 기준).